# Jev API laboratory

Portable extraction of the original interactive tutorial. **Offline by default.** No credentials, original execution outputs, cloud machines or local DSCO checkout are required. Model probabilities are not established calibrated correctness probabilities.

In [ ]:
from pathlib import Path
import sys, os
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from jev_lab.safety import live_enabled
RUN_LIVE = live_enabled()
print('Live API calls:', RUN_LIVE)
# Live calls send cell inputs to TypeSafe and may incur charges.
# Explicit opt-in: export JEV_LAB_LIVE=1 and TYPESAFE_API_KEY before launching Jupyter.


## 1 · Setup

`jev` reads `TYPESAFE_API_KEY` from the environment (it loads a `.env` via python-dotenv
when the variable isn't already set — so a `.env` file beside the kernel works).
The default client targets the `jev-latest` model. This cell is the only one that doesn't
call the API; everything after it does.

In [ ]:
from typing import Literal
from enum import Enum
from pydantic import BaseModel, Field
import jev
print('Jev imported; live examples are explicit opt-in.')

## 2 · `@jev.fn` — the docstring *is* the state

The body-less form: the docstring is rendered as a Jinja2 template with the bound
arguments and sent as the state. `return fn.state()` type-checks the body as
`-> ReturnModel` while leaving the docstring as the whole state (a bare `...` or
`raise NotImplementedError` works too, but `state()` is unambiguous).

In [ ]:
if RUN_LIVE:
    class Sentiment(BaseModel):
        label: Literal["positive", "negative", "neutral"]
        is_strong: bool

    @jev.fn
    def sentiment(review: str) -> Sentiment:
        """Classify the sentiment of this product review:

        {{ review }}
        """
        return sentiment.state()

    r = sentiment("Absolutely loved it — battery lasts forever and the screen is gorgeous!")
    print(r)
    r2 = sentiment("It arrived broken and support never replied.")
    print(r2)
    r3 = sentiment("Mango Margarita Vape? &_&")
    print(r3)
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 3 · One field = one typed question (Noul · Choice · Score)

The field type selects the question type:

| Annotation | Jev question | Coerced back as |
|---|---|---|
| `bool` | **Noul** (p(yes)) | `p >= threshold` (0.5 default) |
| `Literal[...]` / `Enum` | **Choice** | the selected label/member |
| `int` + `Field(ge=, le=)` | **Score** | `lo + round(expected)` |
| `float` + `Field(ge=, le=)` | **Score** | linear interpolation over levels |

Anything else (`str`, nested models, lists, `Optional`) is a `TypeError` **at decoration
time** (see §12).
> **Live limit found while building this notebook:** the API caps a Score at **10 levels** (client compiles up to 256, but the server returns `400 Too many score levels`). So an integer range must satisfy `le - ge <= 9`. The examples use `ge=0, le=9` (10 levels).

In [ ]:
if RUN_LIVE:
    class PrReview(BaseModel):
        risk: Literal["low", "medium", "high"]              # Choice
        tests_adequate: bool                                # Noul
        comments: int = Field(ge=0, le=9)                   # integer Score (10 levels = live max)                  # integer Score
        confidence: float = Field(ge=0.0, le=1.0)           # float Score

    @jev.fn
    def review_pr(diff: str) -> PrReview:
        """Review this pull-request diff:

        {{ diff }}
        """
        return review_pr.state()

    diff = """
    - def divide(a, b): return a / b
    + def divide(a, b):
    +     if b == 0: raise ValueError("b must be non-zero")
    +     return a / b
    + def test_divide_by_zero(): ...
    """
    print(review_pr(diff))
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 4 · `Enum` choices + `Field(description=...)` are the question text

`Field(description=...)` becomes the question's instructions; without one the field name
is humanized (`is_urgent` → "is urgent"). **Write descriptions — they are the questions.**
Integer score levels default to the numbers in range; override them with
`json_schema_extra={"levels": [...]}` (labels must have exactly `ge..le` entries).

In [ ]:
if RUN_LIVE:
    class Priority(Enum):
        P0 = "p0"
        P1 = "p1"
        P2 = "p2"

    class IssueTriage(BaseModel):
        priority: Priority = Field(description="How urgently must engineering act? P0 = drop everything.")
        affects_payment: bool = Field(description="Does this issue affect billing or payments?")
        severity: int = Field(ge=0, le=3,
            description="User-visible severity.",
            json_schema_extra={"levels": ["cosmetic", "minor", "major", "critical"]})

    @jev.fn
    def triage_issue(report: str) -> IssueTriage:
        """Triage this bug report:

        {{ report }}
        """
        return triage_issue.state()

    print(triage_issue("Checkout charges every customer twice since this morning's deploy."))
    print(triage_issue("The settings page footer has a 1px misalignment in dark mode."))
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 5 · Evaluated bodies — build the state yourself

The body always runs. `return fn.state(value)` sends `value` as the state *verbatim*
(the docstring is plain documentation in this form). Loops, conditionals, f-strings —
whatever Python you need. `fn.state` is typed `value -> ReturnModel`, so the body's
`return` still type-checks.

In [ ]:
if RUN_LIVE:
    class Meeting(BaseModel):
        decision_made: bool = Field(description="Did the meeting reach a concrete decision?")
        next_step_owner: Literal["alice", "bob", "carol", "nobody"] = Field(
            description="Who owns the next step?")
        action_items: int = Field(ge=0, le=5, description="How many distinct action items were assigned?")

    @jev.fn
    def analyze_meeting(transcript_lines: list[str]) -> Meeting:
        """Extract the decision, owner, and action-item count from a meeting transcript."""
        numbered = [f"[{i}] {line}" for i, line in enumerate(transcript_lines)]
        return analyze_meeting.state(
            "Analyze this meeting transcript (one numbered line per utterance):\n" + "\n".join(numbered)
        )

    transcript = [
        "Alice: We need to ship the fix this week.",
        "Bob: Agreed. I'll patch the retry logic today.",
        "Carol: And I'll update the status page. We're decided then.",
    ]
    print(analyze_meeting(transcript))
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 6 · `.map` — many items, ONE request

`fn.map(items)` applies the function to each item in a **single** call: items become a
JSON state array, fields become one set of questions per item, and Jev answers all of
them in parallel. Result is `list[Model]` in input order. Each item still runs through
the body machinery, so evaluated bodies run per item and short-circuits (`return Model(...)`)
skip the API for that item.

In [ ]:
if RUN_LIVE:
    class Lang(BaseModel):
        language: Literal["en", "fr", "es", "de", "ja", "other"]
        is_greeting: bool

    @jev.fn
    def detect_language(text: str) -> Lang:
        """Detect the language of this text:

        {{ text }}
        """
        return detect_language.state()

    texts = [
        "Hello, how are you today?",
        "Bonjour, comment ça va ?",
        "Hola, ¿qué tal?",
        "Guten Morgen, wie geht es dir?",
        "こんにちは、元気ですか？",
        "xQ7! zorp nibbler — not really a language.",
    ]
    results = detect_language.map(texts)     # ONE API call for all six
    for t, r in zip(texts, results):
        print(f"{r.language:>6} | greeting={r.is_greeting!s:<5} | {t}")
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 7 · Async — `await` the call, body may `await` too

`@jev.fn` on an `async def` returns an `AsyncJevFn[P, R]`: `await` the call; the body may
itself `await`. `map` works as well (`await fn.map(items)`). Async clients are created
per event loop (connection pools are loop-bound).

In [ ]:
if RUN_LIVE:
    class SpamVerdict(BaseModel):
        is_spam: bool
        category: Literal["phishing", "promo", "scam", "legit"]

    @jev.fn
    async def classify_email(subject: str, body: str) -> SpamVerdict:
        """Classify this email.

        Subject: {{ subject }}
        Body: {{ body }}
        """
        return classify_email.state()

    v = await classify_email(
        "URGENT: Your account has been suspended",
        "Dear customer, verify your password immediately at secure-login.example or lose access.",
    )
    print(v)
    v2 = await classify_email("Re: lunch Friday", "Does noon at the usual place work?")
    print(v2)
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 8 · Class form — `jev.BaseModel.decide(state)`

For "one blob of state in, one struct out" there is a class interface: same field
machinery, no docstring. Fields compile **at class definition** (unsupported types raise
at import). `Model.decide(state)` queries Jev; `await Model.adecide(state)` is the async
form; the plain constructor validates locally and **skips the API** (the mock seam, §11).
Class attributes `__jev_model__` and `__jev_bool_threshold__` pin model and threshold
per class. Deciding is a classmethod so the call stays typed `-> Self` in mypy/pyright.

In [ ]:
if RUN_LIVE:
    class ContractScan(jev.BaseModel):
        has_liability_cap: bool = Field(description="Does the contract cap liability?")
        governing_law: Literal["delaware", "new_york", "california", "other"]
        term_band: int = Field(ge=0, le=9, description="Contract term band 0-9 (0=month-to-month, 9=9+ years); capped at 10 levels by the API.")

    clause = """
    This Agreement shall be governed by the laws of the State of Delaware. The initial term
    is twelve (12) months. In no event shall either party's aggregate liability exceed the
    fees paid in the prior six months.
    """
    result = ContractScan.decide(clause)
    print(result)

    # plain constructor = no API call (validated locally)
    local = ContractScan(has_liability_cap=True, governing_law="delaware", term_band=1)
    print("constructed locally, no API:", local)
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 9 · Function form — `jev.decide(state, PlainModel)`

The same decision on a **plain** `pydantic.BaseModel` — no subclass, no decorator.
`model=` and `bool_threshold=` mirror the class attributes. On a `jev.BaseModel`
subclass it reuses the questions compiled at class definition; plain models compile
per call, so prefer the class form in hot loops.

In [ ]:
if RUN_LIVE:
    class NewsTag(BaseModel):          # plain pydantic model — no jev.BaseModel
        topic: Literal["tech", "sports", "politics", "finance"]
        is_breaking: bool

    headline = "Fed holds rates steady as inflation cools to 2.1%"
    print(jev.decide(headline, NewsTag))
    print(await jev.adecide("Rockets clinch playoff berth in overtime thriller", NewsTag))
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 10 · Tuning — `bool_threshold` and `model=`

Noul → `bool` is thresholded at **0.5** by default. Tune it per function
(`@jev.fn(bool_threshold=0.7)`), per class (`__jev_bool_threshold__ = 0.7`), per call
(`jev.decide(..., bool_threshold=...)`), or globally via the `JEV_BOOL_THRESHOLD` env var
(resolved per call). Pin the model with `@jev.fn(model="jev-latest")`,
`__jev_model__`, or `jev.decide(..., model=...)`; `None` uses the client default
(`jev-latest`).

In [ ]:
if RUN_LIVE:
    class Gate(BaseModel):
        proceed: bool = Field(description="Is it clearly safe to proceed? Answer yes only if unambiguous.")

    @jev.fn(bool_threshold=0.5)
    def gate_lenient(situation: str) -> Gate:
        """{{ situation }}"""
        return gate_lenient.state()

    @jev.fn(bool_threshold=0.95)
    def gate_strict(situation: str) -> Gate:
        """{{ situation }}"""
        return gate_strict.state()

    situation = "The bridge looks mostly fine; one cable seems slightly frayed but it might just be rust staining."
    print("lenient (>=0.5): ", gate_lenient(situation))
    print("strict  (>=0.95):", gate_strict(situation))

    # class-level threshold via __jev_bool_threshold__
    class StrictGate(jev.BaseModel):
        __jev_bool_threshold__ = 0.95
        proceed: bool = Field(description="Is it clearly safe to proceed? Yes only if unambiguous.")

    print("class-pinned 0.95:", StrictGate.decide(situation))
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')

## 11 · The mock seam — test without the API

Three ways to run with **no API call**:

1. **Return a model from the body** — the call short-circuits and returns it.
2. **`builder(fn)(args)`** — runs only the body, returns the state marker.
3. **`state_payload(marker)`** — unwraps the built state for assertions.

(The class form's plain constructor — §8 — is the fourth.)

In [ ]:
from jev import builder, state_payload

class Route(BaseModel):
    queue: Literal["fast", "slow"]

@jev.fn
def route(request: str) -> Route:
    """Route this request: {{ request }}"""
    if request.startswith("TEST:"):                      # mock seam: no API call
        return Route(queue="fast")
    return route.state()

print("short-circuited (no API):", route("TEST: ping"))
if RUN_LIVE:
    print("via Jev:", route("classify this synthetic request only"))

# unit-test the state builder without any API call
@jev.fn
def summarize(doc: str, focus: str) -> Route:   # reuse Route for the demo
    """Summarize focusing on {{ focus }}:

    {{ doc }}"""
    return summarize.state()

marker = builder(summarize)("Jev answers typed questions.", "architecture")
print("state payload:", repr(state_payload(marker)))

## 12 · Type safety — errors at *import*, not at first call

`@jev.fn` rejects a non-`BaseModel` return annotation statically (pyright/mypy strict)
**and** raises `TypeError` at decoration time. Unsupported field types (`str`, nested
models, lists, `Optional`), unbounded scores (no `ge`/`le`), >255 choice options or
>256 score levels, and stringified-label collisions (`Literal[1, "1"]`) all raise
at decoration/class-definition time. (An out-of-range `bool_threshold` raises `ValueError` instead.) The failure surface is import, not production.

In [ ]:
if 'Gate' not in globals():
    class Gate(BaseModel):
        proceed: bool
def expect_typeerror(label, thunk):
    try:
        thunk()
    except TypeError as e:
        print(f"[{label}] TypeError at decoration time: {e}")
    else:
        print(f"[{label}] !! expected TypeError, got none")

# 1. non-BaseModel return annotation
def bad_return():
    @jev.fn
    def f(x: str) -> str:
        """{{ x }}"""
        return f.state()
expect_typeerror("return: str", bad_return)

# 2. unsupported field type (plain str field)
def bad_field():
    class M(BaseModel):
        name: str
    @jev.fn
    def f(x: str) -> M:
        """{{ x }}"""
        return f.state()
expect_typeerror("field: str", bad_field)

# 3. score without ge/le bounds
def bad_score():
    class M(BaseModel):
        n: int
    @jev.fn
    def f(x: str) -> M:
        """{{ x }}"""
        return f.state()
expect_typeerror("score w/o bounds", bad_score)

# 4. stringified-label collision
def bad_labels():
    class M(BaseModel):
        v: Literal[1, "1"]
    @jev.fn
    def f(x: str) -> M:
        """{{ x }}"""
        return f.state()
expect_typeerror("label collision", bad_labels)

# 5. bad bool_threshold raises ValueError (not TypeError) at decoration time
def bad_threshold():
    @jev.fn(bool_threshold=1.5)
    def f(x: str) -> Gate:
        """{{ x }}"""
        return f.state()
try:
    bad_threshold()
    print("[threshold 1.5] !! expected ValueError, got none")
except ValueError as e:
    print(f"[threshold 1.5] ValueError at decoration time: {e}")

## 13 · Escape hatch — full model-reported probabilities via `typesafe_sdk`

`jev` deliberately discards probabilities: `bool` is thresholded, Choice takes argmax,
Score returns the expected value. When you need the calibrated numbers (confidence-gated
routing is the main reason to use Jev), use `typesafe_sdk` directly — one `system_one`
call answers Noul/Choice/Score questions about a state, returning probabilities,
confidence, and token usage. This is the raw API that `jev` compiles down to.

In [ ]:
if RUN_LIVE:
    from typesafe_sdk import TypeSafeClient, Noul, Choice, Score

    with TypeSafeClient() as client:
        r = client.system_one(
            state="I was charged twice. Fix this NOW, this is the third time.",
            questions={
                "billing":  Noul(instructions="Is this about billing?"),
                "tone":     Choice(instructions="What is the customer's tone?",
                                   criteria={"calm": None, "frustrated": None, "furious": None}),
                "urgency":  Score(instructions="How urgent is this?",
                                  criteria=["whenever", "soon", "today", "immediately"]),
            },
        )

    print("model:", r.model, "| usage:", r.usage)
    print("billing p(yes) =", round(r.nouls["billing"].noul, 3))
    print("tone =", r.choices["tone"].choice,
          "| confidence:", round(r.choices["tone"].confidence, 3),
          "| probs:", {k: round(v, 3) for k, v in r.choices["tone"].probabilities.items()})
    print("urgency =", round(r.scores["urgency"].score, 2),
          "| legend:", r.scores["urgency"].legend,
          "| confidence:", round(r.scores["urgency"].confidence, 3))
else:
    print('Live example skipped; set JEV_LAB_LIVE=1 with your own API key to run.')